# Cleaning new data 

The purpose of this dataset is to clean the new added datatset
1. Miami_home_values.csv	30	Miami-labeled Zillow ZIPs	Home-value reference
2. Miami_rental_values.csv	29	Miami-labeled Zillow ZIPs	Zillow rental source/staging
3. Rental-Market.csv	84	Project ZIP allowlist	Rent and HUD benchmark table
4. miami_transit.csv	78	Project ZIPs with transit	Transit feature table
5. rent-trends.csv	74	ZIPs with sufficient ZORI history	Derived rental trend table

## 1. Drop zip code that was not in data/raw/miami_dade_zctas.txt

to ensure consisntent layout , we ensure the dataser matches with the miami_date-actas , whihc cover the area that we are recommending in this project

## 2. Load, validate, and filter all five datasets

ZIP codes are normalized as five-character strings before comparison so that leading zeros are preserved. ZIP validation filters rows only; it does not remove columns.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
NEW_DATA_DIR = DATA_DIR / "cleaned" / "new"
ZCTA_PATH = DATA_DIR / "raw" / "miami_dade_zctas.txt"

DATASETS = {
    "Miami Home Values": {"path": NEW_DATA_DIR / "Miami_home_values.csv", "zip_column": "RegionName"},
    "Miami Rental Values": {"path": NEW_DATA_DIR / "Miami_rental_values.csv", "zip_column": "RegionName"},
    "Rental Market": {"path": NEW_DATA_DIR / "Rental-Market.csv", "zip_column": "zip_code"},
    "Miami Transit": {"path": NEW_DATA_DIR / "miami_transit.csv", "zip_column": "zip_code"},
    "Rent Trends": {"path": NEW_DATA_DIR / "rent-trends.csv", "zip_column": "zip_code"},
}


def normalize_zip_codes(values: pd.Series) -> pd.Series:
    """Normalize ZIP values without converting them to numeric measurements."""
    return (
        values.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(5)
    )


allowed_zips = {
    line.strip()
    for line in ZCTA_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip()
}
allowed_zips = {zip_code.zfill(5) for zip_code in allowed_zips}

if not allowed_zips:
    raise ValueError(f"The Miami-Dade ZIP allowlist is empty: {ZCTA_PATH}")

cleaned_data = {}
validation_summary = []

for dataset_name, config in DATASETS.items():
    path = config["path"]
    zip_column = config["zip_column"]
    frame = pd.read_csv(path, dtype={zip_column: "string"})

    if zip_column not in frame.columns:
        raise ValueError(f"{path.name} is missing its ZIP column: {zip_column}")

    normalized_zips = normalize_zip_codes(frame[zip_column])
    in_scope = normalized_zips.isin(allowed_zips)
    frame[zip_column] = normalized_zips
    filtered_frame = frame.loc[in_scope].copy()

    cleaned_data[dataset_name] = filtered_frame
    validation_summary.append(
        {
            "dataset": dataset_name,
            "original_rows": len(frame),
            "rows_dropped_outside_miami_dade": int((~in_scope).sum()),
            "rows_remaining": len(filtered_frame),
            "original_columns": len(frame.columns),
            "columns_dropped": 0,
            "remaining_columns": len(filtered_frame.columns),
            "out_of_scope_zips": sorted(normalized_zips.loc[~in_scope].dropna().unique()),
        }
    )

validation_summary = pd.DataFrame(validation_summary)
display(validation_summary.drop(columns=["out_of_scope_zips"]))
display(validation_summary[["dataset", "out_of_scope_zips"]])